In [ ]:
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage

import os

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()


model = init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    model_kwargs={"enable_thinking": True},
)

# 定义状态
class AgentState(TypedDict):
    messages: List[BaseMessage]

@tool
def calculate(expression: str) -> str:
    """计算数学表达式，输入如'2+3*4'"""
    return str(eval(expression))

# 绑定工具
model_with_tools = model.bind_tools([calculate])

# 节点1：模型调用
def call_model(state: AgentState):
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# 节点2：工具执行
def execute_tools(state: AgentState):
    last_message = state["messages"][-1]
    tool_messages = []
    for tool_call in last_message.tool_calls:
        tool_result = calculate.invoke(tool_call["args"])
        tool_messages.append(ToolMessage(content=tool_result, tool_call_id=tool_call["id"]))
    return {"messages": tool_messages}

# 构建图
workflow = StateGraph(AgentState)
workflow.add_node("call_model", call_model)
workflow.add_node("execute_tools", execute_tools)
workflow.add_edge(START, "call_model")
workflow.add_conditional_edges(
    "call_model",
    lambda state: "execute_tools" if state["messages"][-1].tool_calls else END
)
workflow.add_edge("execute_tools", "call_model")

# 编译
agent = workflow.compile()

# 运行
result = agent.invoke({"messages": [HumanMessage(content="计算(100+200)*5的结果")]})
print(result["messages"][-1].content)